# 07 — User Engagement and Adoption Analytics

**Sprint 6 | Report-level engagement analysis | Privacy-safe outputs only**

This notebook explains how report-level user engagement metrics are constructed,
what the canonical definitions mean, and how the final engagement status is produced.
It does not display user-level data, user keys, or direct identifiers.


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

# Repository root — works regardless of notebook launch directory
REPO_ROOT = Path(__file__).resolve().parents[1] if "__file__" in dir() else Path.cwd().parent
ANALYTICS_DIR = REPO_ROOT / "outputs" / "analytics"

# Validate root
assert (REPO_ROOT / "src").exists(), f"Expected repo root, got: {REPO_ROOT}"
print(f"Repository root: {REPO_ROOT}")

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## 1. Business Objective

Sprint 6 answers the question: **who is using which reports, how regularly, and is that base growing or shrinking?**

Key outputs:
- A privacy-safe report-user-day mart (`mart_report_user_daily`) with pseudonymous keys only.
- Observation windows (7d, 28d, previous 28d, 90d) anchored to the data as-of date.
- Report-level activity, cohort, frequency, and concentration metrics.
- A deterministic engagement status for each report (`mart_report_engagement`).

This notebook does **not** determine business value, forecast demand, or recommend retiring reports.
Those decisions require context that analytics outputs alone cannot provide.


## 2. Privacy Architecture

All analytics outputs are **report-level**. Individual users are represented only by `user_key`,
a stable pseudonymous surrogate with no direct identifier.

```
Raw usage events (raw-email events)
        |
        v  [privacy_policy.py: strip raw identifiers, validate user_key]
mart_report_user_daily          <- pseudonymous (user_key only)
        |
        v  [aggregate to report-level]
Activity / Cohort / Frequency / Concentration metrics
        |
        v  [apply suppression: groups < MIN_GROUP_SIZE are masked]
mart_report_engagement          <- report-level only; no user_key column
```

**dim_user.csv** maps `user_key to raw email-based identifiers` and is **RESTRICTED**.
It must never be joined into any analytics output or loaded in this notebook.

**Suppression policy**: any metric derived from fewer users than the suppression threshold
(defined in `PrivacyConfig`) is replaced with `NaN` to prevent re-identification.


## 3. Canonical Engagement Definitions

The following definitions are authoritative across all Sprint 6 code.
They are defined in `src/analytics/engagement_definitions.py`.

| Term | Definition |
|------|-----------|
| Active user | A user with ≥ 1 positive-view event in the observation window |
| Active user day | One row in mart_report_user_daily: a (report, user_key, date) triple with daily_views > 0 |
| Returning user | Active on ≥ 2 distinct dates within the window |
| One-time user | Active on exactly 1 date within the window |
| Repeat-view user | More than 1 total view in the window (regardless of distinct dates) |
| Newly adopted user | First ever use date falls within the recent 28-day window; not active in the previous window |
| Retained user | Active in both the recent and previous 28-day windows |
| Reactivated user | Active in the recent window; not active in the previous window; has prior history before the previous window |
| Lapsed user | Active in the previous window but not in the recent window |
| Unclassified recent user | Active recently; not in previous window; cannot be confirmed as new or reactivated due to incomplete history |

**Important distinction: returning vs repeat-view**
- A returning user requires **multiple distinct dates**.
- A repeat-view user only requires **multiple total views** (can all be on one day).


In [ ]:
# Illustration: five views on one date
print("Five views on one date for a single user:")
print("  one-time user     : YES  (active on exactly 1 date)")
print("  returning user    : NO   (only 1 distinct date)")
print("  repeat-view user  : YES  (5 total views > 1)")
print()
print("Three views spread across three separate dates:")
print("  one-time user     : NO")
print("  returning user    : YES  (3 distinct dates >= 2)")
print("  repeat-view user  : YES  (3 total views > 1)")


## 4. Canonical Report-User-Day Mart

`mart_report_user_daily` is the foundational mart for all Sprint 6 user engagement metrics.

**Grain**: one row per (analytics_run_id, report_id, user_key, usage_date) with positive usage.

**Key fields**:
- `user_key` — pseudonymous surrogate; the only user identifier in analytics outputs.
- `daily_views` — total positive-view events for this user-report-date.
- `active_user_day` — boolean flag (always True for rows in this mart).
- `first_report_use_date` / `latest_report_use_date` — report-scoped history per user.
- `lifetime_returned_flag` — True if the user has activity on any date after their first use date.
- `record_valid` — composite validity flag; only valid rows are used for metric computation.


In [ ]:
mart = pd.read_csv(ANALYTICS_DIR / "mart_report_user_daily.csv", parse_dates=["usage_date"])
# Drop any user_key column from display — never show individual user keys
_display = mart.drop(columns=["user_key"], errors="ignore")
print(f"Rows: {len(mart):,}  |  Reports: {mart['report_id'].nunique()}"
      f"  |  Pseudonymous users: {mart['user_key'].nunique() if 'user_key' in mart.columns else 'n/a'}")
print(f"Date range: {mart['usage_date'].min().date()} -> {mart['usage_date'].max().date()}")
print(f"Valid records: {mart['record_valid'].sum() if 'record_valid' in mart.columns else 'n/a'}")

# Distribution of daily_views — report-level aggregates only
views_dist = mart.groupby("report_id")["daily_views"].sum().describe()
print("\nViews per report (lifetime):")
print(views_dist.round(1))


## 5. User-Data Quality

Before computing any engagement metrics, every source event is classified for data quality.
Events with missing, invalid, or prohibited identifiers are excluded from metric computation.

**Quality tiers** (from `classify_source_record_quality`):
- `valid` — passes all checks; used for metric computation.
- `missing_identifier` — user identifier absent; excluded.
- `invalid_identifier` — fails format or uniqueness checks; excluded.
- `prohibited_identifier` — a direct identifier detected in user column; excluded.
- `invalid_report_id`, `invalid_date`, `future_date`, `zero_or_negative_view` — excluded.

The `excluded_user_event_share` field in downstream metrics records how much of each
report's data was excluded, which informs the `user_data_quality_status` classification.


In [ ]:
quality = pd.read_csv(ANALYTICS_DIR / "report_user_data_quality.csv")
print(f"Reports in quality table: {len(quality)}")
print(f"Columns: {list(quality.columns)}")
print("\nData quality status distribution:")
if "data_quality_status" in quality.columns:
    print(quality["data_quality_status"].value_counts().to_string())

print("\nExclusion summary (report-level):")
excl_cols = [c for c in ["source_event_count", "valid_user_event_count",
             "excluded_event_count", "excluded_user_event_share"] if c in quality.columns]
print(quality[excl_cols].describe().round(3).to_string())


## 6. Observation Windows

All engagement metrics are computed within fixed observation windows anchored
to a single `analytics_as_of_date` (the maximum valid usage date in the mart).

| Window | Duration | Purpose |
|--------|----------|---------|
| 7-day | Most recent 7 days | Very recent pulse check |
| 28-day (recent) | Most recent 28 days | Primary engagement window |
| Previous 28-day | 28 days immediately before the recent window | Comparison baseline |
| 90-day | Most recent 90 days | Medium-term trend |
| Previous 90-day | 90 days before the 90d window | Long-term comparison |

Window boundaries are computed deterministically from the data — not from `datetime.now()`.
This ensures the pipeline is fully reproducible given the same mart snapshot.

The `as_of_date_policy` field records whether the as-of date was adjusted for
date-completeness (e.g. partial-day data truncation).


In [ ]:
# Reconstruct window boundaries from the mart
from src.analytics.engagement_windows import EngagementWindowConfig, build_engagement_window_boundaries
import dataclasses

mart_loaded = pd.read_csv(ANALYTICS_DIR / "mart_report_user_daily.csv", parse_dates=["usage_date"])
cfg_win = EngagementWindowConfig()
bounds = build_engagement_window_boundaries(mart_loaded, cfg_win, analytics_run_id="notebook_display")

print(f"Analytics as-of date : {bounds.analytics_as_of_date}")
print(f"7-day window         : {bounds.window_7d_start} -> {bounds.window_7d_end}")
print(f"28-day window        : {bounds.window_28d_start} -> {bounds.window_28d_end}")
print(f"Previous 28-day      : {bounds.previous_28d_start} -> {bounds.previous_28d_end}")
print(f"90-day window        : {bounds.window_90d_start} -> {bounds.window_90d_end}")
print(f"Timezone             : {bounds.analytics_timezone}")
print(f"Completeness policy  : {bounds.as_of_date_policy}")

print("\nTimeline (approximate):")
print("  |--previous-28d--|--prev-28d--|--recent-28d--|")
print(f"  {bounds.previous_90d_start} ... {bounds.previous_28d_start} ... {bounds.window_28d_start} ... {bounds.analytics_as_of_date}")


## 7. Report History Sufficiency

Before computing metrics for a report, the pipeline checks whether there is enough
history to make each window meaningful.

**Sufficiency checks per window**:
- `history_sufficient_7d` — mart has coverage back to the 7d window start.
- `history_sufficient_28d` — mart has coverage for the full 28-day window.
- `comparison_history_sufficient_28d` — mart covers both the recent AND previous 28d windows.
- `history_sufficient_90d` — mart covers the full 90-day window.

If a window is not covered, the corresponding metrics are set to `NaN` rather
than computed from partial data.

**No-activity vs insufficient-history distinction**:
- `has_any_valid_user_activity = False` + sufficient history → report is genuinely inactive.
- `has_any_valid_user_activity = False` + insufficient history → cannot distinguish inactive from unseen.

The `history_sufficiency_status` field summarises which windows are covered.


In [ ]:
# Build sufficiency table for display (report-level, no user data)
from src.analytics.engagement_windows import build_report_history_sufficiency
from src.analytics.report_user_daily import build_report_user_data_quality

quality_df = pd.read_csv(ANALYTICS_DIR / "report_user_data_quality.csv")
dim_report = pd.DataFrame({
    "report_id": mart_loaded["report_id"].unique(),
    "report_name": "Report",
    "report_activation_date": "2023-01-01",
})
suf = build_report_history_sufficiency(dim_report, mart_loaded, quality_df, bounds, cfg_win, "notebook_display")

print(f"Reports in sufficiency table: {len(suf)}")
print("\nHistory sufficiency status distribution:")
print(suf["history_sufficiency_status"].value_counts().to_string())

suf_cols = [c for c in ["report_id", "analytics_as_of_date", "first_observed_usage_date",
            "latest_observed_usage_date", "available_calendar_history_days",
            "history_sufficient_28d", "comparison_history_sufficient_28d",
            "history_sufficient_90d", "history_sufficiency_status"] if c in suf.columns]
print("\nSufficiency summary (first 5 reports):")
print(suf[suf_cols].head(5).to_string(index=False))


## 8. Active-User Breadth

**Breadth** measures how many distinct pseudonymous users accessed a report within each window.

Key fields from `report_user_activity_metrics`:
- `unique_users_28d` — count of distinct `user_key` values with ≥ 1 view in the 28-day window.
- `unique_users_previous_28d` — same for the previous 28-day window (used for trend comparison).
- `active_user_change_28d` / `active_user_change_28d_pct` — absolute and percentage change.
- `active_user_direction_28d` — `growing`, `stable`, or `declining` (or `insufficient_history`).
- `breadth_status` — `broad_adoption` (many users) vs `niche_adoption` (few but regular users).

**Broad vs niche**: a report is classified as niche if it has fewer than 10 active users in 28d
but shows strong returning-user behaviour (high retention share, regular return frequency).
A stable niche audience is not a signal of failure — it may reflect a specialised function.


In [ ]:
from src.analytics.user_engagement_metrics import build_report_user_activity_metrics, UserEngagementMetricsConfig
import dataclasses

bounds_df = pd.DataFrame([dataclasses.asdict(bounds)])
activity = build_report_user_activity_metrics(suf, mart_loaded, quality_df, bounds_df, UserEngagementMetricsConfig(), "notebook_display")

print(f"Reports with activity metrics: {len(activity)}")
print("\nActive-user direction (28d):")
print(activity["active_user_direction_28d"].value_counts().to_string())

# Chart: unique_users_28d by report, coloured by direction
act_plot = activity.dropna(subset=["unique_users_28d"]).sort_values("unique_users_28d", ascending=False)
direction_colors = {"growing": "#2ecc71", "stable": "#3498db", "declining": "#e74c3c",
                    "insufficient_history": "#95a5a6"}
colors = [direction_colors.get(str(d), "#bdc3c7") for d in act_plot["active_user_direction_28d"]]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(act_plot["report_id"], act_plot["unique_users_28d"], color=colors)
ax.set_xlabel("Report ID")
ax.set_ylabel("Unique Users (28d)")
ax.set_title(f"Active-User Breadth by Report (n={len(act_plot)})")
ax.tick_params(axis="x", rotation=45)
# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=k) for k, c in direction_colors.items()]
ax.legend(handles=legend_elements, loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()


## 9. Returning, One-Time, and Repeat-View Users

Three complementary lenses on how users engage with a report:

| Metric | What it measures |
|--------|-----------------|
| `returning_user_share_28d` | Share of active users who visited on ≥ 2 distinct dates in the 28d window |
| `one_time_user_share_28d` | Share of active users who visited on exactly 1 date (complement of returning) |
| `repeat_view_user_share_28d` | Share of active users with > 1 total view (may include same-day multi-views) |

A report can have a high `repeat_view_user_share` but a low `returning_user_share` if users
tend to view the same report multiple times in one session but do not return on other days.

The `repeat_usage_status` field summarises whether a report shows strong, moderate, or low
repeat engagement based on the `returning_user_share_28d` threshold from `EngagementStatusConfig`.


In [ ]:
act_valid = activity.dropna(subset=["returning_user_share_28d", "one_time_user_share_28d"]).sort_values("report_id")

print(f"Reports with returning/one-time split: {len(act_valid)}")
print("\nReturning user share summary:")
print(act_valid["returning_user_share_28d"].describe().round(3).to_string())

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(act_valid))
ax.bar(x, act_valid["returning_user_share_28d"], label="Returning users", color="#3498db")
ax.bar(x, act_valid["one_time_user_share_28d"],
       bottom=act_valid["returning_user_share_28d"], label="One-time users", color="#e67e22")
ax.set_xticks(list(x))
ax.set_xticklabels(act_valid["report_id"], rotation=45, ha="right")
ax.set_ylabel("Share of active users (28d)")
ax.set_title(f"Returning vs One-Time User Share (n={len(act_valid)})")
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
plt.tight_layout()
plt.show()


## 10. Engagement Cohorts

The cohort analysis classifies each active user in the recent 28-day window into one of
four categories by comparing their activity across the recent and previous windows:

| Cohort | Condition |
|--------|-----------|
| `newly_adopted` | First ever use date falls within the recent 28d; no activity in previous window |
| `retained` | Active in both the recent and previous 28d windows |
| `reactivated` | Active recently; not in previous window; has pre-previous history |
| `lapsed` | Active in the previous window; not active in the recent window |
| `unclassified_recent` | Active recently; not in previous window; history too short to classify |

**Denominator note**: lapse rate is computed over the *previous* 28d user base.
Retained and reactivated rates are computed over the *recent* 28d user base.

**Cohort status**: `growing`, `stable`, `high_lapse`, `high_newly_adopted`, or `no_cohort_data`.


In [ ]:
from src.analytics.user_engagement_cohorts import build_report_engagement_cohorts, CohortConfig

cohorts = build_report_engagement_cohorts(suf, mart_loaded, quality_df, bounds_df, CohortConfig(), "notebook_display")

print(f"Reports with cohort data: {len(cohorts)}")
print("\nCohort status distribution:")
print(cohorts["cohort_status"].value_counts().to_string())

# Stacked bar: cohort composition per report
cohort_cols = ["newly_adopted_users_28d", "retained_users_28d", "reactivated_users_28d",
               "unclassified_recent_users_28d"]
cohort_labels = ["Newly Adopted", "Retained", "Reactivated", "Unclassified Recent"]
cohort_colors = ["#2ecc71", "#3498db", "#f39c12", "#95a5a6"]

coh_plot = cohorts.dropna(subset=cohort_cols[:1]).sort_values("report_id").copy()
for col in cohort_cols:
    if col not in coh_plot.columns:
        coh_plot[col] = 0
    coh_plot[col] = coh_plot[col].fillna(0)

fig, ax = plt.subplots(figsize=(10, 4))
bottoms = [0] * len(coh_plot)
x = range(len(coh_plot))
for col, label, color in zip(cohort_cols, cohort_labels, cohort_colors):
    vals = coh_plot[col].tolist()
    ax.bar(x, vals, bottom=bottoms, label=label, color=color)
    bottoms = [b + v for b, v in zip(bottoms, vals)]

ax.set_xticks(list(x))
ax.set_xticklabels(coh_plot["report_id"], rotation=45, ha="right")
ax.set_ylabel("Users (28d)")
ax.set_title(f"Engagement Cohort Composition per Report (n={len(coh_plot)})")
ax.legend(fontsize=9, loc="upper right")
plt.tight_layout()
plt.show()


## 11. Frequency and Intensity

Frequency metrics measure how often and how intensively active users engage with a report.

**Key fields** (from `report_user_frequency_metrics`):

| Field | Definition |
|-------|-----------|
| `views_per_active_user_28d` | Total views in 28d / unique active users |
| `views_per_user_day_28d` | Total views / total user-report-days (adjusts for multi-day users) |
| `median_views_per_user_28d` | Median of per-user view counts (robust to outliers) |
| `median_user_active_days_28d` | Median of per-user distinct active dates |
| `mean_return_gap_days_28d` | Average gap (days) between a returning user's consecutive visits |
| `median_return_gap_days_28d` | Median return gap |

**Percentile interpolation**: p75, p90 are computed using `pandas.quantile(interpolation='linear')`.
Only users with ≥ 2 active dates contribute to return-gap metrics.

**Frequency direction**: `increasing`, `stable`, or `decreasing` based on `views_per_active_user` change.


In [ ]:
from src.analytics.user_frequency_metrics import build_report_frequency_metrics, FrequencyMetricsConfig

frequency = build_report_frequency_metrics(suf, mart_loaded, quality_df, bounds_df, FrequencyMetricsConfig(), "notebook_display")

print(f"Reports with frequency metrics: {len(frequency)}")
print("\nFrequency direction distribution:")
print(frequency["frequency_direction"].value_counts().to_string())

print("\nViews per active user (28d) summary:")
if "views_per_active_user_28d" in frequency.columns:
    print(frequency["views_per_active_user_28d"].describe().round(2).to_string())

# Scatter: views_per_active_user_28d vs unique_users_28d
freq_plot = frequency.dropna(subset=["views_per_active_user_28d", "unique_users_28d"])
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(freq_plot["unique_users_28d"], freq_plot["views_per_active_user_28d"],
           alpha=0.7, edgecolors="grey", linewidths=0.5, color="#3498db")
for _, row in freq_plot.iterrows():
    ax.annotate(row["report_id"], (row["unique_users_28d"], row["views_per_active_user_28d"]),
                fontsize=7, ha="left", va="bottom", alpha=0.7)
ax.set_xlabel("Unique Users (28d)")
ax.set_ylabel("Views per Active User (28d)")
ax.set_title(f"Frequency vs Breadth (n={len(freq_plot)})")
plt.tight_layout()
plt.show()


## 12. Concentration and Dependency

Concentration metrics measure how evenly view activity is spread across users.
A highly concentrated report depends on a small number of heavy users.

**Key fields** (from `report_user_concentration_metrics`):

| Field | Definition |
|-------|-----------|
| `top_1_user_view_share_28d` | Views from the single most active user / total views |
| `top_3_users_view_share_28d` | Views from the top 3 users / total views |
| `top_10pct_users_view_share_28d` | Views from the top 10% of users (ceiling at 5) / total views |
| `user_view_hhi_28d` | Herfindahl–Hirschman Index: sum of squared view-share fractions per user |
| `effective_user_count_28d` | 1 / HHI — the equivalent number of equally active users |
| `effective_user_share_28d` | effective_user_count / unique_users |

**HHI interpretation**: 0 = perfectly equal; 1 = one user dominates all views.
HHI > 0.35 triggers `concentrated_dependency` classification.

**Top-10% ceiling**: to avoid 100% concentration for single-user reports,
the top-10% group is capped at min(ceil(0.1 × N), 5) users.

**Privacy suppression**: HHI and top-user shares are suppressed (set to NaN)
when the active user count falls below the `PrivacyConfig.MIN_GROUP_SIZE` threshold,
as these metrics can effectively identify individuals in small groups.


In [ ]:
from src.analytics.user_concentration_metrics import build_report_concentration_metrics, ConcentrationMetricsConfig

concentration = build_report_concentration_metrics(suf, mart_loaded, quality_df, bounds_df, ConcentrationMetricsConfig(), "notebook_display")

print(f"Reports with concentration metrics: {len(concentration)}")
print("\nConcentration status distribution:")
print(concentration["concentration_status"].value_counts().to_string())
print("\nHHI suppressed (privacy):")
suppressed_hhi = concentration["user_view_hhi_28d"].isna().sum()
print(f"  {suppressed_hhi} of {len(concentration)} reports have HHI suppressed")

# Scatter: HHI vs unique_users (suppressed shown as grey dots at y=0)
fig, ax = plt.subplots(figsize=(8, 4))
not_suppressed = concentration.dropna(subset=["user_view_hhi_28d", "unique_users_28d"])
suppressed = concentration[concentration["user_view_hhi_28d"].isna() & concentration["unique_users_28d"].notna()]

ax.scatter(not_suppressed["unique_users_28d"], not_suppressed["user_view_hhi_28d"],
           alpha=0.8, edgecolors="grey", linewidths=0.5, color="#e74c3c", label="HHI available")
if not suppressed.empty:
    ax.scatter(suppressed["unique_users_28d"], [0] * len(suppressed),
               alpha=0.5, color="#bdc3c7", marker="x", label="HHI suppressed (privacy)")
ax.axhline(0.35, linestyle="--", color="orange", linewidth=1, label="Concentration threshold (0.35)")
ax.set_xlabel("Unique Users (28d)")
ax.set_ylabel("User View HHI (28d)")
ax.set_title(f"Concentration by Report (n={len(concentration)})")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


## 13. Privacy Suppression

The pipeline applies suppression at the metric level — not by removing rows from the mart —
to ensure all reports appear in the final output while protecting small-group privacy.

**Suppression logic**:
1. If a report has fewer than `PrivacyConfig.MIN_GROUP_SIZE` active users in a window,
   concentration and cohort metrics derived from that small group are set to `NaN`.
2. If a report has *any* excluded events due to `prohibited_identifier_event_count > 0`,
   the whole report is flagged with `privacy_limitation_issue`.
3. The `privacy_suppression_status` field records whether suppression was applied.
4. The `privacy_suppressed_fields` column lists which specific fields were suppressed.

**What suppression does not change**:
- `unique_users_28d` is preserved (a count, not a share).
- `overall_engagement_status` is set to `privacy_limited` when suppression prevents classification.
- All suppressed reports still appear in `mart_report_engagement` with a non-null status.

**This ensures**: no report silently disappears from outputs, and no metric can be used to
infer individual user identities from a small group.


In [ ]:
from src.analytics.report_engagement_mart import build_report_engagement_mart
from src.analytics.report_engagement_status import EngagementStatusConfig

eng_mart = build_report_engagement_mart(
    suf, activity, cohorts, frequency, concentration, quality_df, bounds_df,
    EngagementStatusConfig(), "notebook_display"
)

print("Privacy suppression summary:")
print(f"  Reports with any suppression : {eng_mart['activity_privacy_suppressed'].sum()}")
print(f"  privacy_suppression_status   :")
print(eng_mart["privacy_suppression_status"].value_counts().to_string())
print(f"\nPrivacy-limited reports (overall_engagement_status):")
pl = eng_mart[eng_mart["overall_engagement_status"] == "privacy_limited"]
print(f"  Count: {len(pl)}")
if not pl.empty and "privacy_reasons" in pl.columns:
    print(pl[["report_id", "unique_users_28d", "privacy_reasons"]].to_string(index=False))


## 14. Canonical Report Engagement Mart

`mart_report_engagement` is the final, report-level engagement output of Sprint 6.

**Grain**: one row per report_id (for the current analytics run).

**Key columns** (see `MART_REPORT_ENGAGEMENT_COLS` in `report_engagement_mart.py`):
- Identity: `report_id`, `report_name`, `analytics_as_of_date`
- Breadth: `unique_users_28d`, `active_user_direction_28d`, `breadth_status`
- Retention: `returning_user_share_28d`, `lapse_rate_28d`, `cohort_status`
- Frequency: `views_per_active_user_28d`, `median_return_gap_days_28d`, `frequency_status`
- Concentration: `user_view_hhi_28d`, `top_1_user_view_share_28d`, `concentration_status`
- Status: `overall_engagement_status`, `primary_engagement_issue`, `recommended_engagement_action`
- Evidence: `engagement_evidence_status`, `privacy_suppression_status`

This table contains **no user_key column** and no individual-level data.


In [ ]:
print(f"Reports in mart: {len(eng_mart)}")

# Status distributions
for col in ["overall_engagement_status", "breadth_status", "repeat_engagement_status",
            "engagement_evidence_status", "recommended_engagement_action"]:
    if col in eng_mart.columns:
        print(f"\n{col}:")
        print(eng_mart[col].value_counts().to_string())

# Compact display table
display_cols = [c for c in [
    "report_id", "unique_users_28d", "active_user_direction_28d",
    "returning_user_share_28d", "cohort_status", "frequency_status",
    "concentration_status", "overall_engagement_status",
    "primary_engagement_issue", "recommended_engagement_action",
    "engagement_evidence_status"
] if c in eng_mart.columns]
display_cols = [c for c in display_cols if c != "user_key"]
print("\nReport engagement summary:")
print(eng_mart[display_cols].to_string(index=False))


In [ ]:
# Chart 1: overall_engagement_status counts (horizontal bar)
status_counts = eng_mart["overall_engagement_status"].value_counts().sort_values()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].barh(status_counts.index, status_counts.values, color="#3498db")
axes[0].set_xlabel("Report count")
axes[0].set_title(f"Engagement Status Distribution (n={len(eng_mart)})")

# Chart 2: evidence status
if "engagement_evidence_status" in eng_mart.columns:
    ev_counts = eng_mart["engagement_evidence_status"].value_counts()
    axes[1].bar(ev_counts.index, ev_counts.values, color="#e67e22")
    axes[1].set_ylabel("Report count")
    axes[1].set_title("Evidence Status Distribution")
    axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


## 15. Engagement Classification Logic

The `overall_engagement_status` is determined by a **priority-ordered hierarchy**
of issue checks. Each check is computed independently, then the highest-priority
triggered issue determines the status.

**Priority order** (highest first):
1. `no_valid_user_data` — report has no valid user events at all.
2. `privacy_limited` — too few users for any meaningful classification.
3. `insufficient_evidence` — history too short for the primary windows.
4. `inactive` — no users in the last 28 days; but prior history exists.
5. `newly_active` — report first seen in the last 14 days.
6. `declining_adoption` — user count dropped materially (≥ 20%) vs previous 28d.
7. `elevated_lapse` — lapse rate ≥ 40% of previous period's users.
8. `concentrated_dependency` — HHI > 0.35 (high dependency on few users).
9. `low_repeat_usage` — returning_user_share < 25%.
10. `growing_adoption` — user count grew materially (≥ 20%).
11. `healthy_broad_adoption` — many users, good returning share.
12. `healthy_niche_adoption` — few users but consistent returning behaviour.
13. `stable_engagement` — all other passing reports.

**Recommended action** maps from status to a standardised intervention recommendation.
No action recommends retiring, deleting, or restricting access to reports.


In [ ]:
# Show issue flags and their relationship to the final status
issue_cols = [c for c in eng_mart.columns if c.endswith("_issue")]
print("Issue flags in mart:")
for col in issue_cols:
    flagged = eng_mart[col].fillna(False).sum()
    if flagged > 0:
        print(f"  {col}: {flagged} reports flagged")

print("\nIssue count distribution:")
if "engagement_issue_count" in eng_mart.columns:
    print(eng_mart["engagement_issue_count"].value_counts().sort_index().to_string())

print("\nAction priority distribution:")
if "engagement_action_priority" in eng_mart.columns:
    print(eng_mart["engagement_action_priority"].value_counts().to_string())


## 16. Representative Case Studies

The following examples illustrate each engagement status category using actual
reports from the current run. Reports are selected deterministically (sorted by `report_id`).

No individual user data or user keys are shown.


In [ ]:
case_statuses = [
    "healthy_broad_adoption", "healthy_niche_adoption", "growing_adoption",
    "declining_adoption", "elevated_lapse", "inactive", "privacy_limited",
    "insufficient_evidence", "no_valid_user_data",
]

display_cols = [c for c in [
    "report_id", "unique_users_28d", "active_user_direction_28d",
    "overall_engagement_status", "primary_engagement_issue",
    "recommended_engagement_action", "engagement_evidence_status",
    "engagement_reasons"
] if c in eng_mart.columns]

for status in case_statuses:
    subset = eng_mart[eng_mart["overall_engagement_status"] == status]
    if subset.empty:
        print(f"\n[{status}] -- no examples available in current data")
        continue
    ex = subset.sort_values("report_id").iloc[0]
    print(f"\n--- {status.upper()} ---")
    for col in display_cols:
        val = ex.get(col, "n/a")
        print(f"  {col}: {val}")


## 17. Relationship to Sprint 7 Report Analytics

Sprint 6 produces **user-level engagement metrics aggregated to report level**.
Sprint 7 (planned) will add **report-level content and adoption analytics** that join
engagement data with report metadata, forecasting results, and performance telemetry.

**Data flow**:
```
Sprint 6 (this notebook)
  mart_report_engagement → sprint7_report_analytics_mart
                         → Streamlit dashboard (future)
                         → GenAI insights context (future)
```

**What Sprint 7 will add**:
- Report age, content complexity, and page-depth signals.
- Forecast trend overlaid with engagement direction.
- Cross-report benchmarking.
- Aggregated portfolio-level summaries.

**What Sprint 6 does NOT do**:
- It does not connect to the Streamlit app.
- It does not invoke the GenAI layer.
- It does not modify forecasting outputs.
- It does not make retirement or deletion recommendations.


## 18. Fields for Later Consumers

The following fields in `mart_report_engagement` are specifically designed as
inputs for downstream consumers (Sprint 7 analytics, Streamlit, GenAI):

| Field | Consumer | Purpose |
|-------|---------|---------|
| `overall_engagement_status` | All | Primary classification label |
| `recommended_engagement_action` | Streamlit / stakeholders | Standardised next step |
| `engagement_reasons` | GenAI | Plain-language explanation of status |
| `primary_engagement_issue` | GenAI / Streamlit | Top issue driving the status |
| `engagement_evidence_status` | Data quality monitoring | Flags insufficient history |
| `breadth_status` | Sprint 7 | Niche vs broad audience classification |
| `repeat_engagement_status` | Sprint 7 | Returning-user characterisation |
| `review_required` | Ops / data owners | Boolean flag for manual review queue |
| `engagement_action_priority` | Streamlit | High / medium / low priority sorting |

All fields are **report-level**. No field exposes individual user keys, emails, or identities.


## 19. Limitations

1. **Synthetic data**: outputs are based on simulated usage data and do not reflect real
   Power BI telemetry. Metric distributions may not match production patterns.

2. **Pseudonymous keys only**: `user_key` is a stable surrogate with no cross-session linkage
   beyond what is in the mart. If the same real user has multiple keys, they are counted separately.

3. **No business-value signal**: engagement volume does not indicate business importance.
   A report with few users may be critical; a high-traffic report may be redundant.
   This notebook cannot determine which reports should be prioritised or retired.

4. **Window completeness**: metrics are suppressed (NaN) when history coverage is incomplete.
   Reports first activated within the last 28 days will have no comparison metrics.

5. **Suppression threshold**: small-group suppression may mask genuine engagement signals for
   reports with narrow, specialised audiences. These appear as `privacy_limited`.

6. **No user-journey stitching**: the mart tracks per-report user activity.
   It does not capture cross-report user journeys or multi-report sessions.

7. **Privacy-first trade-offs**: by design, no individual-level rows or user keys appear
   in the final mart. This makes the outputs safe to share but limits drill-down capability.
